In [1]:
import sqlite3

def update_experiment_view(db_path='../cem_results.db'):
    """
    Drops the existing experiment_summary view and recreates it 
    with the updated metrics schema including the testset column.
    """
    # Define the SQL script
    sql_script = """
    DROP VIEW IF EXISTS experiment_summary;

    CREATE VIEW experiment_summary AS
    SELECT 
        r.dataset,
        r.entity, 
        r.train_size, 
        r.seed, 
        r.lm,
        m.pollution, 
        m.iteration, 
        m.testset,
        m.f1_score,
        m.precision,
        m.recall,
        m.is_final,
        r.run_id
    FROM metrics m
    JOIN runs r ON m.run_id = r.run_id;
    """

    try:
        # Establish connection
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Execute the script (handles multiple statements)
        cursor.executescript(sql_script)
        
        conn.commit()
        print("View 'experiment_summary' has been updated successfully.")
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if conn:
            conn.close()

# Run the update


In [2]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Connection
conn = sqlite3.connect('../cem_results.db')
update_experiment_view()
i = 50 # Set your desired number of recent runs here

try:
    # 2. Query the last 'i' run_ids from the "runs" table
    # We order by timestamp to ensure we get the truly most recent runs
    id_query = f"""SELECT r.run_id FROM runs r
    WHERE EXISTS (
    SELECT 1 FROM metrics m 
    WHERE m.run_id = r.run_id
    AND m.is_final = 1 
    AND m.testset = 'conv'
    )
    ORDER BY r.timestamp DESC LIMIT {i}"""
    last_run_ids = pd.read_sql_query(id_query, conn)['run_id'].tolist()
    
    if not last_run_ids:
        print("No runs found in the database.")
    else:
        print(last_run_ids)
        # 3. Query experiment_summary for these IDs where is_final=1
        # Using placeholders to prevent SQL injection and handle the list safely
        placeholders = ', '.join(['?'] * len(last_run_ids))
        summary_query = f"""
            SELECT * FROM experiment_summary 
            WHERE run_id IN ({placeholders}) 
            AND is_final = 1 AND testset = 'conv' AND dataset = 'music' ORDER BY run_id DESC
        """
        
        # Execute and load into DataFrame
        df_results = pd.read_sql_query(summary_query, conn, params=last_run_ids)
        
        print(f"Successfully retrieved data for {len(last_run_ids)} runs.")
        display(df_results)

finally:
    conn.close()

View 'experiment_summary' has been updated successfully.
['9762008_2_0940', '9761784_1_2115', '9761782_3_2115', '9761785_2_2115', '9761783_0_2115', '9761777_3_2038', '9761779_1_2038', '9761780_2_2038', '9761778_0_2038', '9761760_3_1940', '9761762_1_1940', '9761761_0_1940', '9761763_2_1940']
Successfully retrieved data for 13 runs.


,dataset,entity,train_size,seed,lm,pollution,iteration,testset,f1_score,precision,recall,is_final,run_id
0,music,track,0.125,42,roberta,high,1,conv,0.947,0.957,0.937,1,9762008_2_0940
1,music,track,0.125,42,roberta,high,1,conv,0.224,0.661,0.135,1,9762008_2_0940
2,music,track,0.125,42,roberta,high,1,conv,0.869,1.000,0.768,1,9762008_2_0940
3,music,track,0.125,42,roberta,high,1,conv,0.473,0.354,0.714,1,9762008_2_0940
4,music,track,0.125,42,roberta,high,1,conv,0.696,0.566,0.904,1,9762008_2_0940
5,music,track,0.125,42,roberta,high,1,conv,0.000,0.000,0.000,1,9762008_2_0940
6,music,track,0.125,42,roberta,high,1,conv,0.400,0.250,1.000,1,9762008_2_0940
7,music,track,0.125,42,roberta,high,1,conv,0.400,0.250,1.000,1,9762008_2_0940
8,music,track,0.125,42,roberta,high,1,conv,0.862,0.906,0.823,1,9762008_2_0940
9,music,track,0.125,42,roberta,high,1,conv,0.400,0.250,1.000,1,9762008_2_0940


In [14]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Establish Connection
# Replace 'experiments.db' with your actual database file path
conn = sqlite3.connect('../cem_results.db')

# 2. Execute Query for the Latest Run
query = """
SELECT *
FROM experiment_summary
WHERE run_id = '9761785_2_2115' 
AND is_final = 1;
"""
# WHERE run_id = (SELECT MAX(run_id) FROM experiment_summary);

df_latest = pd.read_sql_query(query, conn)

# 3. Quick Data Inspection
print(f"Total rows in latest run ({df_latest['run_id'].iloc[0]}): {len(df_latest)}")
display(df_latest)

conn.close()

Total rows in latest run (9761785_2_2115): 4


,dataset,entity,train_size,seed,lm,pollution,iteration,f1_score,precision,recall,is_final,run_id
0,imdb,movie,0.125,42,roberta,high,2,0.570,0.430,0.848,1,9761785_2_2115
1,imdb,movie,0.125,42,roberta,high,2,0.594,0.430,0.961,1,9761785_2_2115
2,imdb,movie,0.125,42,roberta,high,2,0.159,0.087,0.951,1,9761785_2_2115
3,imdb,movie,0.125,42,roberta,high,2,0.473,0.315,0.945,1,9761785_2_2115


9761785_2_2115
9761784_1_2115
9761783_0_2115
9761782_3_2115
9761780_2_2038
9761779_1_2038
9761778_0_2038
9761777_3_2038
9761763_2_1940
9761762_1_1940
9761761_0_1940
9761760_3_1940
9761759_2_1920
9761758_1_1920
9761757_0_1920
9761756_3_1920
9761744_2_1659
9761743_1_1659
9761742_0_1659
9761741_3_1659
